# Dos líneas base y cinco familias
Se entrenan primero DummyClassifier y la regla desgaste >= 200 o torque >= 60; luego logística, Random Forest, HistGradientBoosting, MLP y GaussianNB con valores por defecto salvo semilla. Misma preparación y tres folds para todos.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from src.datos import DOCS, split
from src.pipeline import make_pipeline, PhysicalFeatures


In [2]:
from src.experimentos import baseline_candidates
rows = baseline_candidates()
display(pd.DataFrame(rows).drop(columns=["fold_ap","params"]))

,name,phase,run_id,cv_ap_mean,cv_ap_std,train_ap,fold_train_ap_mean,cv_cost_per_1000,fit_seconds,cv_fit_seconds,gap_ap,warnings,learning_rate
0,dummy,baseline,ce60cc0838084d2e9b1fbb20cdf2d504,0.033875,0.000213,0.033875,0.033875,270999.883454,0.009024,0.031591,1.456823e-08,0,None
1,business_rule,baseline,c1bd50d09be44c93bbd4f41b5f5e2173,0.135011,0.028939,0.134074,0.134297,189627.482799,0.001063,0.004519,-9.368807e-04,0,None
2,logistic,candidato,7a4ebb4ac06c4414ba85910448265712,0.498286,0.069881,0.505527,0.506151,221000.975778,0.013154,0.035740,7.240666e-03,0,None
3,random_forest,candidato,9b518295ce8a40e98bcb622f9c48aac7,0.868115,0.041762,1.000000,1.000000,129812.425922,0.600027,1.313889,1.318851e-01,0,None
4,hist_boosting,candidato,5dac6d2cbb32455aad5a72c456841c89,0.878331,0.035684,1.000000,1.000000,121312.691315,0.128927,0.376170,1.216691e-01,0,0.1
5,mlp,candidato,dd89b550da4f4111b5af7f67f5edbdb0,0.753560,0.052316,0.868367,0.858493,164127.482002,1.725824,3.434191,1.148073e-01,4,constant
6,gaussian_nb,candidato,816e02646d994180847176189245deca,0.379369,0.019116,0.376087,0.380010,193688.865598,0.008974,0.026494,-3.282343e-03,0,None


In [3]:
ablation=json.loads((DOCS/'ablacion.json').read_text(encoding='utf-8'))
default=next(r for r in rows if r['name']=='hist_boosting')
display({'AP_con_features':default['cv_ap_mean'],'AP_sin_features':ablation['cv_ap_mean'], 'delta':default['cv_ap_mean']-ablation['cv_ap_mean'],'std_con_features':default['cv_ap_std']})

{'AP_con_features': 0.8783308704569471,
 'AP_sin_features': 0.8021225832900815,
 'delta': 0.07620828716686567,
 'std_con_features': 0.03568409005261371}

La brecha train AP menos CV AP ayuda a diagnosticar sobreajuste. La regla de negocio es una hipótesis del PSet anterior, pendiente de validación de un técnico. Las corridas se recuperan del checkpoint al reejecutar el notebook, sin duplicar búsqueda ni consumir el holdout.

In [4]:
from src.llm import run as prompting
display(prompting())

{'status': 'executed',
 'model': 'ChatGPT web (versión interna no visible)',
 'date': '2026-11-20',
 'rows': 60,
 'positive_rows': 3,
 'threshold': 0.5,
 'metrics': {'ap': 0.1410569105691057,
  'roc_auc': 0.6929824561403509,
  'recall': 0.0,
  'precision': 0.0,
  'f1': 0.0,
  'cost_per_1000': 400000.0,
  'alert_rate': 0.0,
  'tp': 0,
  'fp': 0,
  'fn': 3,
  'tn': 57},
 'dataset_sha256': 'dbdb2601fec1714c56141c3b24833f6fb671342cadb68d516238a58cbb9f9e1e',
 'protocol': '3 chats independientes. 6 ejemplos etiquetados y 20 consultas ciegas por fold. Solo train.',
 'scope': 'Prueba exploratoria. La muestra pequeña no interviene en la selección del champion.',
 'run_id': '006d039ede6d4ba0994c72dda6e38817'}

El problema tabular puede formularse mediante prompting; se entrega un adaptador real para ChatGPT o Claude. Si no hay API key y modelo configurados, la salida indica not_executed y no afirma una evaluación inexistente. La prueba usa solo submuestras de folds de entrenamiento y nunca el holdout. Su presupuesto distinto no altera el ranking principal.

## Evaluaciones adicionales de modelos preentrenados

In [5]:
import json
from pathlib import Path
for name in ("tabpfn.json", "llm.json"):
    path = Path("docs") / name
    if path.is_file():
        display(json.loads(path.read_text(encoding="utf-8")))

{'source': 'snowflake',
 'dataset_sha256': 'dbdb2601fec1714c56141c3b24833f6fb671342cadb68d516238a58cbb9f9e1e',
 'checkpoint': 'https://raw.githubusercontent.com/automl/TabPFN/tabpfn_v1/tabpfn/models_diff/prior_diff_real_checkpoint_n_0_epoch_42.cpkt',
 'checkpoint_sha256': '3c9aadaeddbf51462af8c0ee4b3ca3c697890f77e92318abbb0821b75261c392',
 'package': 'tabpfn==0.1.11',
 'scope': 'CV en train; contexto estratificado de 768 filas por fold; sin ajuste de pesos.',
 'metrics': {'cv_ap_mean': 0.5672673028785388,
  'cv_ap_std': 0.09966721128257545,
  'cv_cost_per_1000': 204876.06903600667},
 'folds': [{'fold': 1,
   'context': 768,
   'validation': 2667,
   'fit_seconds': 0.0013605369999964978,
   'inference_seconds': 15.114913673999993,
   'ap': 0.6608296843463867,
   'roc_auc': 0.9556159185961282,
   'recall': 0.4444444444444444,
   'precision': 0.7692307692307693,
   'f1': 0.5633802816901409,
   'cost_per_1000': 182227.22159730035,
   'alert_rate': 0.01949756280464942,
   'tp': 40,
   'fp':

{'status': 'executed',
 'model': 'ChatGPT web (versión interna no visible)',
 'date': '2026-11-20',
 'rows': 60,
 'positive_rows': 3,
 'threshold': 0.5,
 'metrics': {'ap': 0.1410569105691057,
  'roc_auc': 0.6929824561403509,
  'recall': 0.0,
  'precision': 0.0,
  'f1': 0.0,
  'cost_per_1000': 400000.0,
  'alert_rate': 0.0,
  'tp': 0,
  'fp': 0,
  'fn': 3,
  'tn': 57},
 'dataset_sha256': 'dbdb2601fec1714c56141c3b24833f6fb671342cadb68d516238a58cbb9f9e1e',
 'protocol': '3 chats independientes. 6 ejemplos etiquetados y 20 consultas ciegas por fold. Solo train.',
 'scope': 'Prueba exploratoria. La muestra pequeña no interviene en la selección del champion.',
 'run_id': '006d039ede6d4ba0994c72dda6e38817'}